# EduVision_DV — Data Cleaning
### Milestone 1 deliverable: `education_cleaning.ipynb`

Input: `university_raw_data.csv` (9,800 rows, 4 real ranking sources stacked, genuinely uncleaned)
Output: `university_cleaned.csv` (one consolidated, cleaned, standardized file)

This notebook cleans the raw consolidated file produced by `data_collection.py`. Every
transformation is shown with a before/after check, not just applied silently.

## 1. Load and inspect

In [1]:
import pandas as pd
import numpy as np
import re

raw = pd.read_csv("university_raw_data.csv")
print("Shape:", raw.shape)
print("\nSource counts:")
print(raw["source"].value_counts())
print("\nMissing values (top 10 columns):")
print(raw.isna().sum().sort_values(ascending=False).head(10))

Shape: (9800, 40)

Source counts:
source
ARWU      4897
THE       2603
CWUR      2200
QS2023     100
Name: count, dtype: int64

Missing values (top 10 columns):
Female:Male Ratio              9712
Teaching Score                 9700
Citations Score                9700
International Outlook Score    9700
International Student          9700
OverAll Score                  9700
Research Score                 9700
No of student per staff        9700
No of student                  9700
Industry Income Score          9700
dtype: int64


## 2. See the actual messiness before fixing it
Real problems in this real data — not hypothetical examples.

In [2]:
print("Country spelling inconsistency:")
print(raw[raw['country'].astype(str).str.contains('United States|USA', na=False)]['country'].unique())

print("\nNumbers stored as text (THE source):")
print(raw[raw.source=='THE'][['num_students','international_students']].dropna().head(3))

print("\nDuplicate (name, year) rows within a single source:")
dupe_check = raw[raw.source=='ARWU']
print(dupe_check.duplicated(subset=['name','year']).sum(), "duplicate rows in ARWU alone")

Country spelling inconsistency:
<StringArray>
['United States of America', 'USA', 'United States']
Length: 3, dtype: str

Numbers stored as text (THE source):
  num_students international_students
0       20,152                    25%
1        2,243                    27%
2       11,074                    33%

Duplicate (name, year) rows within a single source:
6 duplicate rows in ARWU alone


## 3. Standardize text fields
Trim whitespace, collapse casing inconsistencies, and map known country
name variants to one canonical spelling.

In [3]:
def std_text(v):
    if pd.isna(v):
        return None
    return " ".join(str(v).strip().split())

COUNTRY_ALIASES = {
    "United States of America": "United States", "USA": "United States",
    "UK": "United Kingdom", "Russian Federation": "Russia",
    "Republic of Korea": "South Korea", "Hong Kong": "Hong Kong SAR",
}

def std_country(v):
    v = std_text(v)
    if v is None:
        return None
    return COUNTRY_ALIASES.get(v, v)

raw["name"] = raw["name"].apply(std_text)
raw["country"] = raw["country"].apply(std_country)

print("Country spellings after standardization:")
print(raw[raw['country'] == 'United States']['country'].unique(), "-- collapsed to one value")

Country spellings after standardization:
<StringArray>
['United States']
Length: 1, dtype: str -- collapsed to one value


## 4. Parse messy numeric fields
Percent signs, comma-thousands separators, and 'a : b' ratio strings all
need to become real numbers before any KPI math can use them.

In [4]:
def pct_to_float(v):
    if pd.isna(v):
        return np.nan
    return pd.to_numeric(str(v).replace('%','').replace(',','').strip(), errors='coerce')

def ratio_first_half(v):
    if pd.isna(v):
        return np.nan
    s = str(v)
    if ':' in s:
        try:
            return float(s.split(':')[0].strip())
        except ValueError:
            return np.nan
    return pct_to_float(v)

for col in ["num_students", "international_students", "No of student", "International Student"]:
    if col in raw.columns:
        raw[col] = raw[col].apply(pct_to_float)

if "Female:Male Ratio" in raw.columns:
    raw["Female:Male Ratio"] = raw["Female:Male Ratio"].apply(ratio_first_half)

for col in ["rank", "total_score", "score", "OverAll Score"]:
    if col in raw.columns:
        raw[col] = raw[col].astype(str).str.extract(r"([\d\.]+)").astype(float)

print("num_students after parsing (THE):")
print(raw[raw.source=='THE']['num_students'].dropna().head(3).tolist())

num_students after parsing (THE):
[20152.0, 2243.0, 11074.0]


## 5. Remove duplicates
A duplicate is the same (name, year) inside the SAME source — not across
sources, since different sources legitimately cover the same university.

In [5]:
before = len(raw)
raw = raw.drop_duplicates(subset=["source", "name", "year"])
print(f"{before} -> {len(raw)} rows ({before - len(raw)} duplicates removed)")

9800 -> 9794 rows (6 duplicates removed)


## 6. Add an entity-resolution key
This doesn't merge rows across sources yet (that's the next pipeline
stage) — it just adds a normalized key so the SAME real university is
identifiable across sources later, even when spelled slightly differently
("The University of Tokyo" vs "University of Tokyo").

In [6]:
def norm_key(name):
    if pd.isna(name):
        return None
    s = str(name).lower().strip()
    s = re.sub(r"^the\s+", "", s)
    s = re.sub(r"[^a-z0-9]", "", s)
    return s

raw["university_key"] = raw["name"].apply(norm_key)
print("Distinct universities across all 4 sources (by normalized key):", raw["university_key"].nunique())

Distinct universities across all 4 sources (by normalized key): 1374


## 7. Data quality summary

In [7]:
total_cells = raw.size
missing_cells = raw.isna().sum().sum()
completeness = (1 - missing_cells/total_cells) * 100

print(f"Total rows: {len(raw):,}")
print(f"Total columns: {len(raw.columns)}")
print(f"Distinct universities (by key): {raw['university_key'].nunique():,}")
print(f"Missing cells: {missing_cells:,} / {total_cells:,}")
print(f"Overall completeness: {completeness:.1f}%")
print()
print("Completeness is intentionally uneven across columns -- each source")
print("only publishes ITS OWN metrics (e.g. only CWUR has 'quality_of_faculty').")
print("That's a real characteristic of the data, not a cleaning defect.")

Total rows: 9,794
Total columns: 41
Distinct universities (by key): 1,374
Missing cells: 265,405 / 401,554
Overall completeness: 33.9%

Completeness is intentionally uneven across columns -- each source
only publishes ITS OWN metrics (e.g. only CWUR has 'quality_of_faculty').
That's a real characteristic of the data, not a cleaning defect.


## 8. Save the cleaned, consolidated file

In [8]:
lead_cols = ["source", "university_key", "name", "country", "year", "rank"]
lead_cols = [c for c in lead_cols if c in raw.columns]
other_cols = [c for c in raw.columns if c not in lead_cols]
cleaned = raw[lead_cols + other_cols].sort_values(["university_key", "year", "source"]).reset_index(drop=True)

cleaned.to_csv("university_cleaned.csv", index=False)
print(f"Wrote university_cleaned.csv: {cleaned.shape[0]} rows, {cleaned.shape[1]} columns")

Wrote university_cleaned.csv: 9794 rows, 41 columns


## Next step
`university_cleaned.csv` feeds into Milestone 2's `generate_education_kpis.py`,
which resolves `university_key` into a single `university_id` per real
institution, computes the 6 required KPIs, and produces
`university_final_dataset.xlsx`.